# Network Policies

NetworkPolicies erlauben ausgewählten Traffic. Voraussetzung: Netzwerk-Plugin mit Policy-Durchsetzung (K3s standardmäßig mit Network Policy Controller, sofern nicht deaktiviert).

## Voraussetzungen

Python-Kernel, kubectl und K3s-Trainingscluster. Aus diesem Kapitelverzeichnis ausführen. Die verwendeten Demo-Ressourcen müssen frei sein. Aktiven Kontext und Namespace prüfen.

In [ ]:
%%bash
set -eo pipefail
kubectl config current-context
NS=$(kubectl config view --minify -o jsonpath='{..namespace}')
test "${NS:-default}" = default
kubectl get nodes

## 1. Ausgangssituation

Neuen Namespace anlegen; bei bestehendem Namespace abbrechen. Beide Clients sollen den Webserver erreichen.

In [ ]:
%%bash
set -eo pipefail
kubectl create namespace security-net-demo
kubectl apply -f pods.yml
kubectl -n security-net-demo wait --for=condition=Ready pod --all --timeout=180s

In [ ]:
%%bash
set -eo pipefail
for P in allowed denied; do kubectl -n security-net-demo exec "$P" -- wget -T 5 -qO- http://web/; done

## 2. Ingress einschränken

Nur Pods mit access=allowed im gleichen Namespace dürfen TCP-Port 80 erreichen.

In [ ]:
%%bash
set -eo pipefail
cat policy.yml
kubectl apply -f policy.yml
sleep 5
kubectl -n security-net-demo exec allowed -- wget -T 5 -qO- http://web/

In [ ]:
%%bash
set -eo pipefail
if kubectl -n security-net-demo exec denied -- wget -T 5 -qO- http://web/; then echo 'Policy noch nicht wirksam oder nicht durchgesetzt'; exit 1; else echo 'Zugriff fehlgeschlagen; Policy und Netzwerkstatus prüfen'; fi

## 3. Aufräumen

Nur den neu erzeugten Namespace löschen.

In [ ]:
%%bash
set -eo pipefail
kubectl delete namespace security-net-demo

## Fazit

NetworkPolicies begrenzen Netzwerkzugriffe, ersetzen aber weder Authentifizierung noch TLS.